# Formal-to-Casual Rewriter

## 1. Project Overview

**Task:** Change text tone from formal to casual while preserving the original meaning. We explore rule-based substitution, LLM rewriting, and tone-controlled generation across multiple registers (casual, friendly, conversational, humorous).

**Approach:** Start with rule-based replacements as a baseline, then use LLM prompting with tone specifications, and evaluate meaning preservation using the LLM as a semantic judge.

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` — no API keys required.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Implement rule-based** tone transformation as a baseline |
| 2 | **Control LLM tone** via system prompts |
| 3 | **Preserve meaning** while changing register |
| 4 | Test **multiple tone registers** (casual, friendly, humorous) |
| 5 | **Evaluate** meaning preservation with a semantic judge |

## 3. Problem Statement

Formal text (legal documents, corporate emails, academic papers) is often hard to read. Rewriting it in a casual tone makes it accessible — but the meaning must be preserved exactly.

## 4. Why This Matters

- **Customer communication:** Turn legal terms into friendly explanations
- **Internal docs:** Make policies readable by everyone
- **Content adaptation:** Same message, different audience
- **Tone control** is a core LLM capability used in chatbots, assistants, and content tools

## 5. Setup

In [ ]:
import os, json, re, time, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0.3)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")

## 6. Formal Text Samples

We use texts from different formal domains: corporate, legal, academic, and medical.

In [ ]:
FORMAL_TEXTS = {
    "corporate_email": {
        "category": "Corporate",
        "text": "We would like to inform you that your application has been received and is currently under review. Please be advised that the review process typically requires 5-7 business days. Should you have any questions regarding the status of your application, please do not hesitate to contact our support team.",
    },
    "policy_update": {
        "category": "HR Policy",
        "text": "Effective immediately, all employees are required to submit their expense reports within 30 calendar days of the expenditure date. Failure to comply with this policy may result in the forfeiture of reimbursement. Additionally, receipts exceeding $25.00 must be accompanied by a detailed justification memorandum.",
    },
    "legal_terms": {
        "category": "Legal",
        "text": "The Licensee hereby acknowledges and agrees that the Licensor shall not be held liable for any indirect, incidental, consequential, or punitive damages arising from or related to the use of the Software, regardless of whether such damages were foreseeable or whether the Licensor was advised of the possibility thereof.",
    },
    "academic": {
        "category": "Academic",
        "text": "The empirical findings of this study suggest a statistically significant correlation between regular physical activity and improved cognitive function in adults aged 50 and above, with effect sizes comparable to those observed in pharmacological interventions for mild cognitive impairment.",
    },
    "medical": {
        "category": "Medical",
        "text": "It is recommended that patients presenting with persistent cephalgia of greater than 72 hours duration accompanied by photophobia and emesis should be evaluated for potential intracranial pathology via contrast-enhanced magnetic resonance imaging.",
    },
}

for key, item in FORMAL_TEXTS.items():
    print(f"[{item['category']}] {item['text'][:80]}...")

---

## 7. Approach 1: Rule-Based Substitution

In [ ]:
FORMAL_TO_CASUAL = {
    "we would like to inform you": "just letting you know",
    "please be advised": "heads up",
    "do not hesitate to": "feel free to",
    "should you have any questions": "if you have questions",
    "failure to comply": "if you don't",
    "may result in": "could mean",
    "effective immediately": "starting now",
    "hereby acknowledges and agrees": "agrees",
    "shall not be held liable": "isn't responsible",
    "regardless of whether": "even if",
    "it is recommended that": "you should",
    "presenting with": "who have",
    "in the event that": "if",
    "at this point in time": "now",
    "in order to": "to",
    "due to the fact that": "because",
    "for the purpose of": "to",
    "in spite of the fact that": "although",
    "accompanied by": "with",
    "regarding the status of": "about",
}

def rule_based_rewrite(text):
    result = text
    changes = []
    for formal, casual in FORMAL_TO_CASUAL.items():
        if formal.lower() in result.lower():
            result = re.sub(re.escape(formal), casual, result, flags=re.IGNORECASE)
            changes.append(f"  \u2192 '{formal}' \u2192 '{casual}'")
    return result, changes

print("RULE-BASED REWRITING")
print("=" * 60)
for key, item in list(FORMAL_TEXTS.items())[:3]:
    rewritten, changes = rule_based_rewrite(item["text"])
    print(f"\n[{item['category']}]")
    print(f"  Original: {item['text'][:100]}...")
    print(f"  Rewrite:  {rewritten[:100]}...")
    print(f"  Changes:  {len(changes)}")
    for c in changes: print(f"  {c}")

## 8. Approach 2: LLM-Based Rewriting

In [ ]:
CASUAL_SYSTEM = """Rewrite formal text in a casual, friendly tone.

Rules:
- Keep ALL the same information and meaning
- Use everyday language instead of jargon
- Use contractions (don't, we'll, it's)
- Keep sentences shorter
- Make it sound like a helpful friend explaining something
- Do NOT add new information
- Do NOT remove important details
- Return ONLY the rewritten text"""

print("LLM CASUAL REWRITING")
print("=" * 60)
llm_results = {}
for key, item in FORMAL_TEXTS.items():
    rewritten = ask(CASUAL_SYSTEM, f"Formal text:\n{item['text']}")
    llm_results[key] = rewritten
    print(f"\n[{item['category']}]")
    print(f"  FORMAL:  {item['text']}")
    print(f"  CASUAL:  {rewritten}")

## 9. Multiple Tone Registers

Same formal text rewritten in different casual registers.

In [ ]:
TONES = {
    "Friendly professional": "Rewrite in a friendly but professional tone. Like a helpful colleague.",
    "Super casual": "Rewrite very casually. Like texting a friend. Use slang where natural.",
    "Empathetic": "Rewrite with warmth and empathy. Acknowledge the reader's feelings.",
    "Gen-Z": "Rewrite in Gen-Z communication style. Use modern internet language naturally.",
}

test_text = FORMAL_TEXTS["corporate_email"]["text"]
print(f"FORMAL: {test_text}")
print("=" * 60)

for tone_name, instruction in TONES.items():
    sys = f"{instruction}\nPreserve ALL information. Return ONLY the rewritten text."
    result = ask(sys, f"Rewrite this:\n{test_text}")
    print(f"\n{tone_name}:")
    print(f"  {result}")

## 10. Meaning Preservation Check

The most important evaluation: does the casual version mean the same thing?

In [ ]:
JUDGE_SYSTEM = """You compare a formal text with its casual rewrite.
Check:
1. meaning_preserved: Is all factual information kept? (1-5, 5=identical meaning)
2. tone_shift: How much did the tone change? (1-5, 5=completely casual)
3. info_lost: Any important details dropped? List them.
4. info_added: Any new info invented? List them.

Return JSON: {"meaning_preserved": N, "tone_shift": N, "info_lost": [], "info_added": [], "verdict": "pass|fail"}"""

def parse_judge(text):
    text = clean(text)
    if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
    s, e = text.find("{"), text.rfind("}") + 1
    if s >= 0 and e > s:
        try: return json.loads(text[s:e])
        except: pass
    return None

print("MEANING PRESERVATION CHECK")
print("=" * 60)
for key, item in FORMAL_TEXTS.items():
    if key not in llm_results: continue
    resp = ask(JUDGE_SYSTEM, f"Formal:\n{item['text']}\n\nCasual:\n{llm_results[key]}")
    scores = parse_judge(resp)
    if scores:
        print(f"\n[{item['category']}]")
        print(f"  Meaning: {scores.get('meaning_preserved','?')}/5 | Tone shift: {scores.get('tone_shift','?')}/5")
        print(f"  Verdict: {scores.get('verdict','?')}")
        if scores.get("info_lost"): print(f"  Lost: {scores['info_lost']}")
        if scores.get("info_added"): print(f"  Added: {scores['info_added']}")

## 11. Reverse Direction: Casual to Formal

In [ ]:
FORMAL_SYSTEM = """Rewrite this casual text in formal, professional language.
Use proper grammar, formal vocabulary, and professional tone.
Preserve all meaning. Return ONLY the rewritten text."""

casual_samples = [
    "Hey, just wanted to let you know your order shipped! Should get there by Friday.",
    "We messed up on the last update and a bunch of users lost their saved data. We're fixing it ASAP.",
    "The app's been kinda slow lately because we've got way more users than we expected. Working on it!",
]

print("CASUAL \u2192 FORMAL")
print("=" * 60)
for text in casual_samples:
    formal = ask(FORMAL_SYSTEM, f"Casual text:\n{text}")
    print(f"\n  Casual: {text}")
    print(f"  Formal: {formal}")

## 12. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Losing critical details in rewrite | Always check meaning preservation |
| Adding info not in the original | Instruct "do not add new information" |
| Inconsistent tone within one rewrite | Specify the exact register |
| Rule-based only handles known phrases | Use LLM for open-ended tone shifts |

| # | Takeaway |
|---|----------|
| 1 | **Rule-based** catches common formal phrases but misses context-dependent tone |
| 2 | **LLM rewriting** handles nuanced tone shifts naturally |
| 3 | **Meaning preservation** is the hardest part — always verify with a judge |
| 4 | **Multiple registers** (friendly, casual, empathetic) serve different audiences |
| 5 | **Bidirectional** rewriting (casual\u2192formal) uses the same techniques |
| 6 | **Legal/medical** text is hardest to casualize without losing precision |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*